# 9.3 批处理优化 (Batch Processing)

批处理优化通过高效组织推理请求来提升GPU利用率和吞吐量，是推理服务性能的核心决定因素。

本节涵盖：
- Static vs Continuous Batching
- Continuous Batching调度器实现
- Prefill与Decode阶段
- Chunked Prefill
- 调度策略与SLO保障

## 1. Static vs Continuous Batching

**目的**：理解两种批处理方式的区别和性能差异

**Static Batching**：
- 一次性组成batch，所有请求一起开始、一起结束
- 短请求必须等待长请求完成（padding浪费）
- GPU利用率低，浪费可达50%+

**Continuous Batching**（Iteration-level scheduling）：
- 每个decode iteration都可以加入新请求或移除完成的请求
- 完成的请求立即释放资源，新请求立即加入
- GPU利用率高，吞吐量提升2-4x

**关键指标**：
- **TTFT（Time To First Token）**：从请求到首token的延迟
- **TPOT（Time Per Output Token）**：每个输出token的延迟
- **吞吐量**：tokens/sec/GPU
- **SLO达标率**：P99延迟满足服务等级目标的请求比例

In [ ]:
import random

random.seed(42)

n_requests = 8
output_lengths = [random.randint(5, 25) for _ in range(n_requests)]
max_len = max(output_lengths)

print('=== Static vs Continuous Batching ===')
print(f'Requests: {n_requests}')
print(f'Output lengths: {output_lengths}')
print(f'Max length: {max_len}')

static_total = max_len * n_requests
static_useful = sum(output_lengths)
static_waste = static_total - static_useful

print(f'\n--- Static Batching ---')
print(f'Total GPU slots: {static_total} ({max_len} iters × {n_requests} requests)')
print(f'Useful compute: {static_useful}')
print(f'Wasted (padding): {static_waste} ({static_waste/static_total:.1%})')
print(f'GPU utilization: {static_useful/static_total:.1%}')

active = list(range(n_requests))
remaining = output_lengths.copy()
continuous_steps = 0
continuous_compute = 0
completed_at = {}
while active:
    continuous_steps += 1
    continuous_compute += len(active)
    done = []
    for i in active:
        remaining[i] -= 1
        if remaining[i] == 0:
            done.append(i)
            completed_at[i] = continuous_steps
    for d in done:
        active.remove(d)

print(f'\n--- Continuous Batching ---')
print(f'Total GPU slots: {continuous_compute}')
print(f'Useful compute: {continuous_compute} (no padding waste)')
print(f'GPU utilization: 100%')
print(f'Efficiency gain: {static_total/continuous_compute:.1f}x')

print(f'\n--- Per-Request Completion Time ---')
for i in range(n_requests):
    static_time = max_len
    cont_time = completed_at[i]
    print(f'  Request {i} (len={output_lengths[i]:2d}): static={static_time} iters, continuous={cont_time} iters ({cont_time/static_time:.0%})')

avg_static = max_len
avg_cont = sum(completed_at.values()) / n_requests
print(f'\nAvg completion: static={avg_static}, continuous={avg_cont:.1f} ({avg_cont/avg_static:.0%})')
print(f'\nKey: Continuous batching eliminates padding waste and reduces avg completion time.')
print(f'Short requests finish earlier, long requests are not delayed.')

## 2. Continuous Batching调度器

**目的**：实现完整的Continuous Batching调度器，包含KV Cache管理

**调度器核心逻辑**：
1. 每个iteration检查是否有完成的请求（生成EOS或达到max_tokens）
2. 移除完成的请求，释放KV Cache
3. 从等待队列中选择新请求加入batch
4. 选择条件：有足够的KV Cache空间

**KV Cache管理**：
- 每个请求的KV Cache大小 = seq_len × d_model × 2 × n_layers × bytes_per_elem
- PagedAttention将KV Cache分成固定大小的block，按需分配
- 请求完成后释放所有block，供新请求使用

In [ ]:
import random
from collections import deque

random.seed(42)

class ContinuousBatchScheduler:
    def __init__(self, max_batch_size=8, total_kv_blocks=200, block_size=16):
        self.max_batch_size = max_batch_size
        self.total_kv_blocks = total_kv_blocks
        self.block_size = block_size
        self.free_blocks = total_kv_blocks
        self.waiting_queue = deque()
        self.running_requests = []
        self.next_id = 0
        self.completed_results = []

    def submit(self, prompt_len, max_new_tokens, priority=0):
        req_id = self.next_id
        self.next_id += 1
        self.waiting_queue.append({
            'id': req_id, 'prompt_len': prompt_len,
            'max_new_tokens': max_new_tokens,
            'generated': 0, 'phase': 'prefill',
            'kv_blocks': 0, 'priority': priority,
        })
        return req_id

    def _blocks_needed(self, prompt_len, max_new_tokens):
        total_len = prompt_len + max_new_tokens
        return (total_len + self.block_size - 1) // self.block_size

    def schedule(self):
        while self.waiting_queue and len(self.running_requests) < self.max_batch_size:
            req = self.waiting_queue[0]
            needed = self._blocks_needed(req['prompt_len'], req['max_new_tokens'])
            if needed <= self.free_blocks:
                self.waiting_queue.popleft()
                req['kv_blocks'] = needed
                self.free_blocks -= needed
                self.running_requests.append(req)
            else:
                break

    def step(self):
        completed = []
        for req in self.running_requests:
            req['generated'] += 1
            if req['generated'] >= req['max_new_tokens'] or random.random() < 0.05:
                completed.append(req)

        for req in completed:
            self.running_requests.remove(req)
            self.free_blocks += req['kv_blocks']
            self.completed_results.append({
                'id': req['id'], 'generated': req['generated'],
                'prompt_len': req['prompt_len'],
            })

        self.schedule()
        return completed

scheduler = ContinuousBatchScheduler(max_batch_size=4, total_kv_blocks=100)

for i in range(12):
    prompt_len = random.randint(10, 50)
    max_new = random.randint(10, 40)
    scheduler.submit(prompt_len, max_new)

print('=== Continuous Batching Scheduler ===')
print(f'Max batch: {scheduler.max_batch_size}, KV blocks: {scheduler.total_kv_blocks}')

total_steps = 0
total_tokens = 0
while scheduler.running_requests or scheduler.waiting_queue:
    scheduler.schedule()
    if not scheduler.running_requests:
        break
    completed = scheduler.step()
    total_steps += 1
    total_tokens += len(scheduler.running_requests)

    if completed:
        for r in completed:
            print(f'  Step {total_steps}: Request {r["id"]} completed (prompt={r["prompt_len"]}, generated={r["generated"]})')

    if total_steps > 300:
        break

print(f'\nTotal steps: {total_steps}')
print(f'Total tokens generated: {total_tokens}')
print(f'Completed requests: {len(scheduler.completed_results)}/{scheduler.next_id}')
print(f'Throughput: {total_tokens/total_steps:.1f} tokens/step')
print(f'\nKey: Scheduler manages batch composition and KV cache allocation per iteration.')
print(f'New requests enter when KV cache is available; completed requests release blocks immediately.')

## 3. Prefill与Decode阶段

**目的**：理解推理的两个阶段及其不同的计算特性

**Prefill阶段**：
- 处理整个prompt，计算所有token的KV Cache
- 计算密集型：O(seq_len²)的注意力计算
- 生成第一个token（TTFT主要由prefill时间决定）
- 可以利用大batch并行处理多个prompt

**Decode阶段**：
- 每次只处理1个新token，更新KV Cache
- 访存密集型：O(seq_len)的KV Cache读取
- 每步生成1个token（TPOT由decode速度决定）
- batch中不同请求的decode可以并行

**混合batch**：vLLM等框架允许同一batch中同时有prefill和decode请求，但需要特殊的attention kernel

In [ ]:
import torch
import torch.nn as nn
import time

torch.manual_seed(42)

class SimpleAttention(nn.Module):
    def __init__(self, d_model=128, n_heads=4):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, d_model * 3)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x, kv_cache=None):
        B, L, D = x.shape
        qkv = self.qkv(x).reshape(B, L, 3, self.n_heads, self.d_head)
        q, k, v = qkv.unbind(dim=2)

        if kv_cache is not None:
            past_k, past_v = kv_cache
            k = torch.cat([past_k, k], dim=1)
            v = torch.cat([past_v, v], dim=1)

        new_kv_cache = (k, v)
        scale = self.d_head ** -0.5
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale
        attn = attn.softmax(dim=-1)
        out = torch.matmul(attn, v)
        out = out.reshape(B, L, D)
        return self.out(out), new_kv_cache

attn = SimpleAttention(d_model=128, n_heads=4)

print('=== Prefill vs Decode Performance ===')

print(f'\n--- Prefill (process entire prompt) ---')
for prompt_len in [32, 64, 128, 256, 512]:
    x = torch.randn(1, prompt_len, 128)
    for _ in range(3):
        with torch.no_grad():
            _, _ = attn(x)
    start = time.time()
    n_runs = 50
    for _ in range(n_runs):
        with torch.no_grad():
            _, kv = attn(x)
    elapsed = (time.time() - start) / n_runs * 1000
    flops = 2 * prompt_len * prompt_len * 128 * 4 / 1e6
    print(f'  prompt_len={prompt_len:4d}: {elapsed:.2f}ms, ~{flops:.1f} MFLOPs (compute-bound)')

print(f'\n--- Decode (1 new token with KV cache) ---')
prompt = torch.randn(1, 64, 128)
with torch.no_grad():
    _, kv_cache = attn(prompt)

single_token = torch.randn(1, 1, 128)
for _ in range(5):
    with torch.no_grad():
        _, _ = attn(single_token, kv_cache)
start = time.time()
n_runs = 100
for _ in range(n_runs):
    with torch.no_grad():
        _, _ = attn(single_token, kv_cache)
decode_time = (time.time() - start) / n_runs * 1000
print(f'  1 token decode (kv_len=64): {decode_time:.2f}ms (memory-bound)')

print(f'\nKey: Prefill is compute-bound (O(n²)), decode is memory-bound (O(n)).')
print(f'TTFT is dominated by prefill time; TPOT is dominated by decode time.')
print(f'Chunked prefill splits long prompts into chunks to reduce TTFT for concurrent requests.')

## 4. Chunked Prefill与调度策略

**目的**：优化长prompt的prefill，减少对其他请求的影响

**问题**：长prompt的prefill计算量大，会阻塞正在decode的请求，导致TPOT飙升。

**Chunked Prefill**：
- 将长prompt分成多个chunk，每次只处理一个chunk
- 每个chunk后可以切换到decode，保持低TPOT
- 代价：prefill的总时间略增（多次kernel launch开销）

**调度策略**：
- **FCFS（First Come First Served）**：简单公平，但短请求可能等很久
- **SJF（Shortest Job First）**：短请求优先，平均延迟低，但长请求可能饿死
- **优先级调度**：按业务优先级排序，付费用户优先
- **SLO感知调度**：根据请求的SLO deadline动态调整优先级

In [ ]:
import random
from collections import deque

random.seed(42)

class SLOScheduler:
    def __init__(self, max_batch=8, ttft_slo=0.5, tpot_slo=0.1):
        self.max_batch = max_batch
        self.ttft_slo = ttft_slo
        self.tpot_slo = tpot_slo
        self.waiting = deque()
        self.running = []
        self.next_id = 0
        self.current_time = 0.0

    def submit(self, prompt_len, max_new_tokens, arrival_time, priority=0):
        req = {
            'id': self.next_id, 'prompt_len': prompt_len,
            'max_new_tokens': max_new_tokens, 'generated': 0,
            'arrival': arrival_time, 'priority': priority,
            'prefill_done': False, 'ttft': None,
        }
        self.next_id += 1
        self.waiting.append(req)
        return req['id']

    def _slo_urgency(self, req):
        wait_time = self.current_time - req['arrival']
        ttft_deadline = self.ttft_slo - wait_time
        return max(0, 1.0 - ttft_deadline / self.ttft_slo)

    def schedule(self):
        candidates = list(self.waiting)
        candidates.sort(key=lambda r: (-r['priority'], -self._slo_urgency(r)))
        for req in candidates:
            if len(self.running) < self.max_batch:
                self.waiting.remove(req)
                self.running.append(req)

    def step(self, decode_time=0.01, prefill_time_per_token=0.001):
        self.current_time += decode_time
        completed = []
        for req in self.running:
            if not req['prefill_done']:
                req['prefill_done'] = True
                req['ttft'] = self.current_time - req['arrival']
            req['generated'] += 1
            if req['generated'] >= req['max_new_tokens']:
                completed.append(req)

        for req in completed:
            self.running.remove(req)

        self.schedule()
        return completed

scheduler = SLOScheduler(max_batch=4, ttft_slo=0.5, tpot_slo=0.1)

for i in range(10):
    arrival = i * 0.1
    prompt_len = random.randint(20, 100)
    max_new = random.randint(10, 30)
    priority = random.choice([0, 1])
    scheduler.submit(prompt_len, max_new, arrival, priority)

print('=== SLO-Aware Scheduling ===')
print(f'TTFT SLO: {scheduler.ttft_slo}s, TPOT SLO: {scheduler.tpot_slo}s')

all_completed = []
for step in range(500):
    scheduler.schedule()
    completed = scheduler.step()
    for req in completed:
        all_completed.append(req)
    if not scheduler.running and not scheduler.waiting:
        break

print(f'\nCompleted: {len(all_completed)}/{scheduler.next_id}')
ttft_values = [r['ttft'] for r in all_completed if r['ttft'] is not None]
if ttft_values:
    avg_ttft = sum(ttft_values) / len(ttft_values)
    max_ttft = max(ttft_values)
    slo_violations = sum(1 for t in ttft_values if t > scheduler.ttft_slo)
    print(f'Avg TTFT: {avg_ttft:.3f}s, Max TTFT: {max_ttft:.3f}s')
    print(f'SLO violations: {slo_violations}/{len(ttft_values)} ({slo_violations/len(ttft_values):.0%})')

priority_0 = [r for r in all_completed if r['priority'] == 0]
priority_1 = [r for r in all_completed if r['priority'] == 1]
if priority_0 and priority_1:
    avg_ttft_0 = sum(r['ttft'] for r in priority_0) / len(priority_0)
    avg_ttft_1 = sum(r['ttft'] for r in priority_1) / len(priority_1)
    print(f'\nPriority 0 avg TTFT: {avg_ttft_0:.3f}s')
    print(f'Priority 1 avg TTFT: {avg_ttft_1:.3f}s (higher priority = lower TTFT)')

print(f'\nKey: SLO-aware scheduling prioritizes requests close to deadline violation.')
print(f'Higher priority requests get scheduled first.')
print(f'Production systems track SLO compliance as a key metric.')